# Notebook 32 — MEPI v1.5 exact-state convergence continuation

Restart Kernel → Run All. The saved default never trains. v1.5 changes only `max_epochs: 50 → 100`; early stopping remains authoritative. The completed v1.4 directory is immutable and the test subset is unavailable.

In [1]:
import os
import sys

EXPECTED_CUBLAS_WORKSPACE_CONFIG = ":4096:8"
actual = os.environ.get("CUBLAS_WORKSPACE_CONFIG")
print("CUBLAS_WORKSPACE_CONFIG before bootstrap =", actual)
print("torch already imported =", "torch" in sys.modules)
if "torch" in sys.modules:
    raise RuntimeError("Restart kernel: torch was imported before deterministic CUDA bootstrap")
if actual is not None and actual != EXPECTED_CUBLAS_WORKSPACE_CONFIG:
    raise RuntimeError(f"Conflicting CUBLAS_WORKSPACE_CONFIG={actual!r}")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = EXPECTED_CUBLAS_WORKSPACE_CONFIG
print("CUBLAS_WORKSPACE_CONFIG =", os.environ["CUBLAS_WORKSPACE_CONFIG"])


CUBLAS_WORKSPACE_CONFIG before bootstrap = None
torch already imported = False
CUBLAS_WORKSPACE_CONFIG = :4096:8


In [2]:
from pathlib import Path
import json
import torch

torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "configs").is_dir()
print("PROJECT_ROOT =", PROJECT_ROOT)
print("torch =", torch.__version__, "cuda =", torch.cuda.is_available())


PROJECT_ROOT = <REPOSITORY_ROOT>
torch = 2.5.1 cuda = True


In [3]:
from pathlib import Path
import os
import sys

# Find repository root robustly from common notebook launch locations.
_candidates = [
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_ROOT = next(
    (
        p.resolve()
        for p in _candidates
        if (p / "src" / "mepi_v1").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate MEPI project root containing src/mepi_v1. "
        f"Current working directory: {Path.cwd()}"
    )

# Make `from src.mepi_v1...` importable.
project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

# Keep all relative paths deterministic.
os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("WORKING_DIRECTORY =", Path.cwd())
print("PROJECT_ROOT_IN_SYS_PATH =", project_root_str in sys.path)

assert (PROJECT_ROOT / "src" / "mepi_v1" / "config.py").is_file()

PROJECT_ROOT = <REPOSITORY_ROOT>
WORKING_DIRECTORY = <REPOSITORY_ROOT>
PROJECT_ROOT_IN_SYS_PATH = True


In [4]:
from pprint import pprint
from src.mepi_v1.config import load_config
from src.mepi_v1.finetune_v1_5 import (
    PROTOCOL_SHA256, RESUME_NEXT_EPOCH, audit_exact_parent_state,
    audit_v1_5_frozen_evidence, validate_v1_5_config,
)
CONFIG_PATH = PROJECT_ROOT / "configs/finetune_v1_5.yaml"
RUN_DIRECTORY = PROJECT_ROOT / "experiments/finetune_v1_5_xlstm_depth8/baseline_l3_0p3_l4_0p20"
PARENT_RUN = PROJECT_ROOT / "experiments/finetune_v1_4_xlstm_depth8/baseline_l3_0p3_l4_0p20"
config = load_config(CONFIG_PATH)
validate_v1_5_config(config)
frozen_audit = audit_v1_5_frozen_evidence(PROJECT_ROOT, CONFIG_PATH)
parent_audit = audit_exact_parent_state(PARENT_RUN / "last_checkpoint.pt")
provenance = json.loads((RUN_DIRECTORY / "continuation_provenance.json").read_text())
assert provenance["resume_next_epoch"] == RESUME_NEXT_EPOCH == 50
assert config["training"]["max_epochs"] == 100
assert config["training"]["early_stopping_patience"] == 10
assert config["training"]["scheduler"] is None
pprint({"protocol_v1_5_sha256": PROTOCOL_SHA256, "parent_v1_4_state_sha256": provenance["parent_checkpoint_sha256"], "resume_next_epoch": RESUME_NEXT_EPOCH, "max_epochs": 100, "patience": 10, "test_lock": True})


{'max_epochs': 100,
 'parent_v1_4_state_sha256': '22aca18c37b4526c46e3044eeeda4e36a6a0364891e767cace755961eef44138',
 'patience': 10,
 'protocol_v1_5_sha256': '0f68d6ebd5d16177e5238471639f861dfa91522130ac6a4a3ac48634b9af1e39',
 'resume_next_epoch': 50,
 'test_lock': True}


In [ ]:
RUN_TRAINING = False
FORCE_RETRAIN = False
print("RUN_TRAINING =", RUN_TRAINING)
print("FORCE_RETRAIN =", FORCE_RETRAIN)
if FORCE_RETRAIN:
    raise RuntimeError("v1.5 continuation forbids discarding the exact parent state")


RUN_TRAINING = True
FORCE_RETRAIN = False


In [6]:
from src.mepi_v1.finetune_v1_5 import run_v1_5_baseline
finetune_result = None
if RUN_TRAINING:
    finetune_result = run_v1_5_baseline(CONFIG_PATH, RUN_DIRECTORY, force_retrain=FORCE_RETRAIN)
    pprint(finetune_result)
else:
    print("SCIENTIFIC TRAINING: SKIPPED (RUN_TRAINING=False)")
    print("Exact resume state remains prepared at next_epoch=50")


{'best_checkpoint': '<REPOSITORY_ROOT>/experiments/finetune_v1_5_xlstm_depth8/baseline_l3_0p3_l4_0p20/best_checkpoint.pt',
 'best_checkpoint_sha256': '84ab544ab094d602392276effc3fa94ed1a04cb61b214530a438fc710361fd7b',
 'best_epoch': 81,
 'best_validation_total_loss': 0.3626963762676015,
 'epochs_completed': 92,
 'parent_protocol': 'MEPI-FROZEN-PROTOCOL v1.4',
 'parent_state_sha256': '22aca18c37b4526c46e3044eeeda4e36a6a0364891e767cace755961eef44138',
 'resume_next_epoch': 50,
 'resume_source': 'v1.4_exact_parent_state',
 'scheduler': None,
 'status': 'TRAINING_COMPLETE_VALIDATION_ONLY',
 'stopped_early': True,
 'test_accessed': False,
 'test_dataset_created': False,
 'test_loader_created': False}


In [7]:
print("PROTOCOL_V1_5_SHA256 =", PROTOCOL_SHA256)
print("PARENT_V1_4_STATE_SHA256 =", provenance["parent_checkpoint_sha256"])
print("RESUME_NEXT_EPOCH =", RESUME_NEXT_EPOCH)
print("MAX_EPOCHS =", config["training"]["max_epochs"])
print("EARLY_STOPPING_PATIENCE =", config["training"]["early_stopping_patience"])
print("SCIENTIFIC_TRAINING_RUN =", finetune_result is not None)
print("TEST_ACCESSED = FALSE")


PROTOCOL_V1_5_SHA256 = 0f68d6ebd5d16177e5238471639f861dfa91522130ac6a4a3ac48634b9af1e39
PARENT_V1_4_STATE_SHA256 = 22aca18c37b4526c46e3044eeeda4e36a6a0364891e767cace755961eef44138
RESUME_NEXT_EPOCH = 50
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10
SCIENTIFIC_TRAINING_RUN = True
TEST_ACCESSED = FALSE
